**Lab 06 - Klasteryzacja**

Zadanie 1
Celem zadania jest klasteryzacja zbioru zawoerakącego dane dotyczące klientów centrum handlowego z użyciem algorytmu k-means.


Wymagane importy

In [ ]:
import random

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import skfuzzy as fuzz
import numpy as np

Pobranie danych z pliku i wczytanie do zmiennej customers

In [ ]:
customers_raw = pd.read_csv("./customers_mall.csv")
customers = customers_raw["Annual Income;Spending Score"].str.split(";", expand=True)
customers.columns = ["Annual Income", "Spending Score"]
customers = customers.astype(float)

Dane dotyczące klientów składają się z informacji o zarobkach oraz z oceny wydatków klientów w skali od 1 do 100. Aby poprawnie wykonać badanie, należy dokonać normalizacji danych. W tym celu wykorzystano algorym StandardScaler().

In [ ]:
scaler = StandardScaler()
customers_scaled = scaler.fit_transform(customers)

W celu wykonania zadania należy dobrać odpowiednią ilość klastrów. Podstawą dobory w tym zadaniu będzie metoda łokcia. Polega ona na analizie wykresu interia ( odległość punktów od centrów ) w zależności od liczby klastrów.

In [ ]:
inertia = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=0, n_init=10)
    model.fit(customers_scaled)
    inertia.append(model.inertia_)

W celu dokonania wybory liczby klastrów zostanie wygenerowany wykres łokcia, oraz wybrana ilośc klastrów w punkcie w którym wykres zaczybna się spłaszczać

In [ ]:
plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel("Liczba klastrów")
plt.ylabel("Inertia")
plt.title("Metoda łokcia")
plt.grid(True)
plt.show()

Analizując wykres, wybrano ilość klastrów równą 5, ponieważ wtedy wykres zaczął ulegać spłaszczeniu, Dokonano nauczania modelu.

In [ ]:
model = KMeans(n_clusters=5, random_state=0, n_init=10)
model.fit(customers_scaled)
labels = model.labels_
centroids = model.cluster_centers_

W celu poprawniejszego wygnerowania wykresu do analizy, dokonano wycofania normalizacji danych. Następnie wygenerowano wykres

In [ ]:
centroids_original = scaler.inverse_transform(centroids)

plt.scatter(customers["Annual Income"], customers["Spending Score"], c=labels, cmap="viridis")
plt.scatter(centroids_original[:, 0], centroids_original[:, 1], s=300, c='red', marker='X')
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (0-100)")
plt.title("Klasteryzacja klientów centrum handlowego")
plt.grid(True)
plt.show()

Wyniki badania zaprezentowane na powyższej wizualizacji wskazują na dużą poprawność wykonanego zadania. Dane zostały poprawnie rozdzielone na 5 klastrów, gdzie każdy z nich odpowiada za inną grupę klientów. Algorytm KMeans poprawnie poradził sobie z danymi które mają dwa główne wymiary. Oddzielenie pomiędzy klientami o dużych zarobkach i dużej ocenie od klientów o niskich zarobkach i niskiej ocenie jest perfekcyjny. Podział klientów którzy mają niską ocenę również przebiegł pomyślnie, zauważamy że poniżej przychodu na poziomie 57 k$ wszyscy o niskiej ocenie zostali przydzieleni do lewego klastra, natomiast pozostali do tego po prawej. Łatwo zauważyć granicę co również wskazuje na poprawność algorytmu, ale jednocześnie na bardzo dużą poprawność danych wejściowych.

**Zadanie 2**

Celem zadania jest dokonanie klasteryzacji zbioru planet NASA kilkoma sposobami, oraz dokonanie oceny wyników za pomocą odpowiednich metryk.

Zadanie rozpoczęto od wczytania danych, oraz podobnie jak w zadaniu 1 od przeskalowania danych.


In [ ]:
planets_raw = pd.read_csv("planets 1.csv")
planets = planets_raw.drop(columns=["pl_name"])

scaler = StandardScaler()
planets_scaled = scaler.fit_transform(planets)

Badanie zostanie przeprowadzone dla kilku algorytmów dlatego w celu ograniczenia ilości kodu zdefiniowano funkcję do obliczania metryk wymaganych do analizy badania

In [ ]:
def evaluate_clustering(model, data, labels=None):
    if labels is None:
        labels = model.labels_

    if len(set(labels)) > 1:
        silhouette = silhouette_score(data, labels)
        davies_bouldin = davies_bouldin_score(data, labels)
        calinski_harabasz = calinski_harabasz_score(data, labels)
        return silhouette, davies_bouldin, calinski_harabasz
    else:
        return None, None, None

Badanie zostanie przeprowadzone dla 3 metod: KMeans, AgglomerativeClustering, DBSCAN. Dla pierwszej liczba klastrów zostanie dobrana ponownie z wykorzystaniem metody łokcia. Dla drugiego i trzeciego algorytmu dane wejściowe były dobierane ręcznie do momentu uzyskania satysfakcjonującego rezultatu.

Metoda łokcia dla KMeans:

In [ ]:
inertia = []
k_range = range(1, 21)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
    kmeans.fit(planets_scaled)
    inertia.append(kmeans.inertia_)
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, marker='o')
plt.title("Metoda łokcia - KMeans")
plt.xlabel("Liczba klastrów (k)")
plt.ylabel("Inertia")
plt.grid(True)

Na podstawie metody łokcia wybrano liczbę klastrów 13

In [ ]:
results = []

kmeans = KMeans(n_clusters=13, random_state=0, n_init=10)
kmeans.fit(planets_scaled)

silhouette, db_score, ch_score = evaluate_clustering(kmeans, planets_scaled)
results.append(["KMeans", 13, silhouette, db_score, ch_score])

Wywołanie algorymu AgglomerativeClustering

In [ ]:
agg_model = AgglomerativeClustering(n_clusters=10)
agg_model.fit(planets_scaled)
silhouette, db_score, ch_score = evaluate_clustering(agg_model, planets_scaled)
results.append(["AgglomerativeClustering", 10 , silhouette, db_score, ch_score])

Wywołanie algorytmu DBSCAN

In [ ]:
db_model = DBSCAN(eps=3, min_samples=2)
db_labels = db_model.fit_predict(planets_scaled)

if len(set(db_labels)) > 1:
    silhouette, db_score, ch_score = evaluate_clustering(db_model, planets_scaled, db_labels)
    results.append(["DBSCAN", len(set(db_labels)) - (1 if -1 in db_labels else 0), silhouette, db_score, ch_score])
else:
    results.append(["DBSCAN", "No valid clusters", None, None, None])

W celu lepszej wizualizacji wyników, dokonano kowersji danych na DataFrame.

In [ ]:
results_df = pd.DataFrame(results, columns=["Algorithm", "Clusters", "Silhouette Score", "Davies-Bouldin Score", "Calinski-Harabasz Score"])

print(results_df)


Przedstawione rezultaty wskazują, że najlepsze wyniki dla tego badania osiągnął KMeans, mimo dość słabych wyników Silhouette Score. Wysokie Calinski-Harabasz Score wskazuje na to, że KMeans dobrze separuje klastry, natomiast niekoniecznie jest idealnym rozwiązaniem. AgglomerativeClustering osiągnął wyniki trochę słabsze niż KMeans, możliwe że jest to spowodowane złym doborem ilości klastrów.

Zgodnie z poleceniem należało wybrać dowolny z wyników i go opisać. Chciałbym skupić się bardziej na algorytmie DBSCAN. Co prawda nie osiągnął najlepszych wyników, lecz bardzo interesująca jest realizacja tego algorytmu oraz czułość danych wejściowych. Można zauważyć, że dla eps=0.5 oraz min_samples=4, algorytm wygenerował 15 klastrów, jednocześnie osiągając bardzo słabe wyniki wskazujące na niską separowalność między sklastrami, natomiast już dla eps=6.5 wygenerował jeden klaster a wynik się poprawiły. Przy eps=3 i min_samples=14 algorytm wykrył 3 klastry, a wyniki były lepsze. Z kolei dla eps=3 i min_samples=2, DBSCAN wygenerował 5 klastrów z Silhouette Score = 0.7044 i Calinski-Harabasz Score = 77.53, co wskazuje na dobrą separację klastrów. Algorytm radzi sobie z wykrywaniem klastrów o różnej gęstości, ale jego skuteczność zależy w dużej mierze od odpowiednich ustawień parametrów. Parametry powinny być dostosowane do specyfiki danych, aby uzyskać najlepsze wyniki. Optymalne ustawienia eps = 3 i min_samples = 14 dają najlepsze wyniki pod względem jakości klasteryzacji. Wnioskując, DBSCAN jest wrażliwy na parametry, ale przy odpowiednim dopasowaniu może skutecznie wykrywać klastry o różnej gęstości i poprawiać jakość klasteryzacji.

**Zadanie 3**

Celem zadania jest użycie algorytmu fuzzy clustering dla zestawu planet z zadania 2. Jednocześnie wymogiem jest, aby wykorzystać wyłącznie połowę dostępnych kolumn do uczenia.

Wczytano dane, ograniczono liczbę kolumn do pierwszych 5, przeskalowano dane

In [ ]:
planets_raw = pd.read_csv("planets 1.csv")
planets = planets_raw.drop(columns=["pl_name"])
planets_selected = planets.iloc[:, :5]
scaler = StandardScaler()
planets_scaled = scaler.fit_transform(planets_selected)

Wykonanie algorytmu cmeans oraz przygotowanie metryki silhouette do badania

In [ ]:
n_clusters = 4
m = 2.0

cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(planets_scaled.T, c=n_clusters, m=m, error=0.005, maxiter=1000)
fcm_labels = np.argmax(u, axis=0)
silhouette = silhouette_score(planets_scaled, fcm_labels)

Wypisanie wyników, wylosowano 10 punktów dla których wypisano do ilu klastrów należą

In [ ]:
print("Silhouette Score dla Fuzzy C-Means:", silhouette)
for i in range(10):
    rand_point = random.choice(fcm_labels)
    print(f"Punkt {i+1}: Przynależność do klastrów:", rand_point)
print("\nMacierz przynależności (stopień przynależności do każdego klastra):")
print(u[:,:5])


**Zadanie 4**

Celem zadania 4 jest wykorzystanie wszystkich poznanych algorytmów klasteryzacyjnych dla zbioru circle.csv.

Wczytanie danych, sprawdzenie sktruktury i skalowanie danych

In [ ]:
circle_df = pd.read_csv("circle.csv")

scaler = StandardScaler()
circle_scaled = scaler.fit_transform(circle_df)

W celu przemyślenia danych wejściowych, nastąpiło obrazu danych wejściowych

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c='blue', s=30)
plt.title('Wykres punktowy zbioru danych')
plt.xlabel('x1')
plt.ylabel('x2')
plt.grid(True)
plt.show()

Po przeanalizowaniu danych wejściowych, widoczny jest podział danych na dwa pierścienie. Skutkuje to oczywistym wyborem - dla KMeans oraz AgglomerativeClustering wybrane zostaną dwa klastry.

Wykonanie algorytmu KMeans

In [ ]:
results = []

kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
kmeans.fit(circle_scaled)

silhouette, db_score, ch_score = evaluate_clustering(kmeans, circle_scaled)
results.append(["KMeans", 2, silhouette, db_score, ch_score])

Wykonanie algorytmu AgglomerativeClustering

In [ ]:
agg_model = AgglomerativeClustering(n_clusters=2)
agg_model.fit(circle_scaled)
silhouette, db_score, ch_score = evaluate_clustering(agg_model, circle_scaled)
results.append(["AgglomerativeClustering", 2, silhouette, db_score, ch_score])

W celu odpowiedniego wykonania algorytmu DBSCAN należy przeanalizować różne dane wejściowe. Zadanie 2 pokazało że algorytm jest bardzo czuły na te zmiany.

Próba 1:

In [ ]:
db_model = DBSCAN(eps=1, min_samples=5)
db_labels = db_model.fit_predict(circle_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=db_labels, cmap="coolwarm")
plt.title("DBSCAN")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.show()

Próba 2:

In [ ]:
db_model = DBSCAN(eps=5, min_samples=5)
db_labels = db_model.fit_predict(circle_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=db_labels, cmap="coolwarm")
plt.title("DBSCAN")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.show()

Próba 3:

In [ ]:
db_model = DBSCAN(eps=1, min_samples=15)
db_labels = db_model.fit_predict(circle_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=db_labels, cmap="coolwarm")
plt.title("DBSCAN")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.show()

Próba 4:

In [ ]:
db_model = DBSCAN(eps=0.1, min_samples=5)
db_labels = db_model.fit_predict(circle_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=db_labels, cmap="coolwarm")
plt.title("DBSCAN")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.show()

Próba 5:

In [ ]:
db_model = DBSCAN(eps=0.1, min_samples=3)
db_labels = db_model.fit_predict(circle_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=db_labels, cmap="coolwarm")
plt.title("DBSCAN")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.show()

Próba 6:

In [ ]:
db_model = DBSCAN(eps=0.2, min_samples=5)
db_labels = db_model.fit_predict(circle_scaled)

if len(set(db_labels)) > 1:
    silhouette, db_score, ch_score = evaluate_clustering(db_model, circle_scaled, db_labels)
    results.append(["DBSCAN", len(set(db_labels)) - (1 if -1 in db_labels else 0), silhouette, db_score, ch_score])
else:
    results.append(["DBSCAN", "No valid clusters", None, None, None])

plt.figure(figsize=(8, 6))
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=db_labels, cmap="coolwarm")
plt.title("DBSCAN")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.show()

Próby 1, 2, 3 wykazują, że zbut duże wartości epsilon oraz min samples sprawiają, że algorytm bardzo łatwo wyłapuje wszystkie dane jako jeden klaster, co całkowicie blokuje analizę. Podczas próby 4 spróbowano drastycznie zmniejszyć wartości, czego wynikiem okazało się utworzenie wielu klastrów na warstwie drugiej - algorytm zbyt czuły. Próba 5 została wykonana na tej samej wartości epsilon, ale z zmniejszoną liczbą min samples w porównaniu do próby 4. Tutaj ponownie uzyskano zbyt wiele klastrów. W próbie 6 nieznacznie zwiększono wartość epsilon oraz uśredniono min samples na 5. Taki wybór danych wejściowych pozwolił na osiągnięcie oczekiwanego wyniku - dwa klastry rozłożone na pierścienie. Badanie to potwierdza czułość algorytmu DBSCAN -> zmiana zaledwie o 0.1 pozwoliła na perfekcyjne rozwiązania zadania.

Wygenerowano wyniki dla najlepszych prób:

In [ ]:
results_df = pd.DataFrame(results, columns=["Algorithm", "Clusters", "Silhouette Score", "Davies-Bouldin Score", "Calinski-Harabasz Score"])

print(results_df)

plt.figure(figsize=(15, 5))

kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
kmeans.fit(circle_scaled)
labels_kmeans = kmeans.labels_

plt.subplot(1, 3, 1)
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=labels_kmeans, cmap="viridis")
plt.title("KMeans (k=2)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

labels_agg = agg_model.labels_

plt.subplot(1, 3, 2)
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=labels_agg, cmap="plasma")
plt.title("Agglomerative Clustering (k=2)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

plt.subplot(1, 3, 3)
plt.scatter(circle_scaled[:, 0], circle_scaled[:, 1], c=db_labels, cmap="coolwarm")
plt.title("DBSCAN")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

plt.tight_layout()
plt.show()
